In [18]:
import argparse
import os
import time
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

    

import torchvision
import torchvision.transforms as transforms

from models import *   # bring everything in the folder models

In [45]:
cfg = {
    'VGG11': [64, 'M', 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M'],
    'VGG13': [64, 64, 'M', 128, 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M'],
    'VGG16_quant': [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'M', 512, 512, 512, 'M', 512, 512, 512, 'M'],
    'VGG16_quant_custom': ['F', 64, 'M', 128, 128, 'M', 256, 8, 8, 'M', 512, 512, 512, 'M', 512, 512, 512, 'M'],
    'VGG16':              ['F', 64, 'M', 128, 128, 'M', 256, 256, 256, 'M', 512, 512, 512, 'M', 512, 512, 512, 'M'],
    'VGG19': [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 256, 'M', 512, 512, 512, 512, 'M', 512, 512, 512, 512, 'M'],
}


class VGG_quant_custom(nn.Module):
    def __init__(self, vgg_name):
        super(VGG_quant_custom, self).__init__()
        self.features = self._make_layers(cfg[vgg_name])
        self.classifier = nn.Linear(512, 10)

    def forward(self, x):
        out = self.features(x)
        out = out.view(out.size(0), -1)
        out = self.classifier(out)
        return out

    def _make_layers(self, cfg):
        layers = []
        in_channels = 3
        for x in cfg:
            if x == 'M':
                layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
            elif x == 'F':  # This is for the 1st layer
                layers += [nn.Conv2d(in_channels, 64, kernel_size=3, padding=1, bias=False),
                           nn.BatchNorm2d(64),
                           nn.ReLU(inplace=True)]
                in_channels = 64
            else:
                if in_channels == 8 and x == 8:
                    layers += [QuantConv2d(in_channels, x, kernel_size=3, padding=1),
                           nn.ReLU(inplace=True)]
                else:
                    layers += [QuantConv2d(in_channels, x, kernel_size=3, padding=1),
                           nn.BatchNorm2d(x),
                           nn.ReLU(inplace=True)]
                in_channels = x
        layers += [nn.AvgPool2d(kernel_size=1, stride=1)]
        return nn.Sequential(*layers)

    def show_params(self):
        for m in self.modules():
            if isinstance(m, QuantConv2d):
                m.show_params()
    

def VGG16_quant_custom(**kwargs):
    model = VGG_quant_custom(vgg_name = 'VGG16_quant_custom', **kwargs)
    return model

In [48]:
global best_prec
use_gpu = torch.cuda.is_available()
print('=> Building model...')
    
batch_size = 128

model_name = "VGG16"

model = VGG16_quant_custom()

normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ]))
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)


test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ]))

testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print_freq = 100

def train(trainloader, model, criterion, optimizer, epoch):
    batch_time = AverageMeter()   ## at the begining of each epoch, this should be reset
    data_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    model.train()

    end = time.time()  # measure current time
    
    for i, (input, target) in enumerate(trainloader):
        # measure data loading time
        data_time.update(time.time() - end)  # data loading time

        input, target = input.cuda(), target.cuda()

        # compute output
        output = model(input)
        loss = criterion(output, target)

        # measure accuracy and record loss
        prec = accuracy(output, target)[0]
        losses.update(loss.item(), input.size(0))
        top1.update(prec.item(), input.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end) # time spent to process one batch
        end = time.time()


        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   epoch, i, len(trainloader), batch_time=batch_time,
                   data_time=data_time, loss=losses, top1=top1))

def validate(val_loader, model, criterion ):
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    # switch to evaluate mode
    model.eval()

    end = time.time()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):
         
            input, target = input.cuda(), target.cuda()

            # compute output
            output = model(input)
            loss = criterion(output, target)

            # measure accuracy and record loss
            prec = accuracy(output, target)[0]
            losses.update(loss.item(), input.size(0))
            top1.update(prec.item(), input.size(0))

            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if i % print_freq == 0:  # This line shows how frequently print out the status. e.g., i%5 => every 5 batch, prints out
                print('Test: [{0}/{1}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   i, len(val_loader), batch_time=batch_time, loss=losses,
                   top1=top1))

    print(' * Prec {top1.avg:.3f}% '.format(top1=top1))
    return top1.avg


def accuracy(output, target, topk=(1,5)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk) # 5
    batch_size = target.size(0) # 128

    _, pred = output.topk(maxk, 1, True, True) # topk(k, dim=None, largest=True, sorted=True)
                                    # will output (max value, its index)
    pred = pred.t()               # transpose
    correct = pred.eq(target.reshape(1, -1).expand_as(pred))   # "-1": calculate automatically

    res = []
    for k in topk: # 1, 5
        correct_k = correct[:k].reshape(-1).float().sum(0)  # view(-1): make a flattened 1D tensor
        res.append(correct_k.mul_(100.0 / batch_size))   # correct: size of [maxk, batch_size]
    return res


class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n    ## n is impact factor
        self.count += n
        self.avg = self.sum / self.count

        
def save_checkpoint(state, is_best, fdir):
    filepath = os.path.join(fdir, 'checkpoint.pth')
    torch.save(state, filepath)
    if is_best:
        shutil.copyfile(filepath, os.path.join(fdir, 'model_best.pth.tar'))


def adjust_learning_rate(optimizer, epoch):
    """For resnet, the lr starts from 0.1, and is divided by 10 at 80 and 120 epochs"""
    adjust_list = [10, 30, 80, 150, 250]
    if epoch in adjust_list:
        for param_group in optimizer.param_groups:
            param_group['lr'] = param_group['lr'] * 0.1

=> Building model...


In [49]:
lr = 0.01

weight_decay = 5e-4
epochs = 300
best_prec = 0

model = model.cuda()
criterion = nn.CrossEntropyLoss().cuda()
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
# optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

if not os.path.exists('result'):
    os.makedirs('result')
    
fdir = 'result/'+str(model_name)

if not os.path.exists(fdir):
    os.makedirs(fdir)
        

for epoch in range(0, epochs):
    train(trainloader, model, criterion, optimizer, epoch)
    
    # evaluate on test set
    print("Validation starts")
    prec = validate(testloader, model, criterion)

    scheduler.step()

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec,best_prec)
    print('best acc: {:1f}'.format(best_prec))
    save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model.state_dict(),
        'best_prec': best_prec,
        'optimizer': optimizer.state_dict(),
    }, is_best, fdir)

Epoch: [0][0/391]	Time 0.478 (0.478)	Data 0.069 (0.069)	Loss 2.5707 (2.5707)	Prec 7.031% (7.031%)
Epoch: [0][100/391]	Time 0.020 (0.026)	Data 0.001 (0.001)	Loss 1.8766 (2.0205)	Prec 28.906% (27.220%)
Epoch: [0][200/391]	Time 0.020 (0.023)	Data 0.000 (0.001)	Loss 1.6831 (1.8563)	Prec 33.594% (32.229%)
Epoch: [0][300/391]	Time 0.020 (0.022)	Data 0.001 (0.001)	Loss 1.4945 (1.7360)	Prec 45.312% (36.335%)
Validation starts
Test: [0/79]	Time 0.369 (0.369)	Loss 1.5294 (1.5294)	Prec 45.312% (45.312%)
 * Prec 47.120% 
best acc: 47.120000
Epoch: [1][0/391]	Time 0.487 (0.487)	Data 0.068 (0.068)	Loss 1.3904 (1.3904)	Prec 47.656% (47.656%)
Epoch: [1][100/391]	Time 0.020 (0.025)	Data 0.001 (0.001)	Loss 1.2452 (1.3306)	Prec 54.688% (51.276%)
Epoch: [1][200/391]	Time 0.020 (0.023)	Data 0.000 (0.001)	Loss 1.0936 (1.3014)	Prec 54.688% (52.414%)
Epoch: [1][300/391]	Time 0.020 (0.022)	Data 0.001 (0.001)	Loss 1.2221 (1.2671)	Prec 53.906% (53.683%)
Validation starts
Test: [0/79]	Time 0.393 (0.393)	Loss 1.00

In [68]:
model = VGG16_quant_custom()

PATH = "result/VGG16/model_best.pth.tar"
checkpoint = torch.load(PATH)
model.load_state_dict(checkpoint['state_dict'])
device = torch.device("cuda") 

model.cuda()
model.eval()

captured_inputs = []
captured_outputs = []

def hook_fn1(module, inputs):
    captured_inputs.append(inputs[0].detach())
    
def hook_fn2(module, inputs, outputs):
    captured_outputs.append(outputs[0].detach())
    
model.layer1[0].conv1.register_forward_pre_hook(hook_fn1)
model.layer1[0].conv1.register_forward_hook(hook_fn2)

test_loss = 0
correct = 0

with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device) # loading to GPU
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)  
        correct += pred.eq(target.view_as(pred)).sum().item()

test_loss /= len(testloader.dataset)

print('\nTest set: Accuracy: {}/{} ({:.0f}%)\n'.format(
        correct, len(testloader.dataset),
        100. * correct / len(testloader.dataset)))



Test set: Accuracy: 8948/10000 (89%)



In [69]:
conv_input = None
def hook_fn(module, inputs):
    conv_input = inputs[0].detach()
model.layer1[0].conv1.register_forward_pre_hook(hook_fn)

test_input = torch.rand(1, 3, 32, 32)
model(test_input.cuda())

tensor([[ -2.7012, -23.3942,  -3.9263,   0.6204,  -0.5026, -11.4387,   0.1560,
         -16.5292,  -9.1900, -11.2488]], device='cuda:0',
       grad_fn=<AddmmBackward0>)

In [70]:
print(model)

ResNet_Cifar(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): QuantConv2d(
        16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
        (weight_quant): weight_quantize_fn()
      )
      (conv2): QuantConv2d(
        16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
        (weight_quant): weight_quantize_fn()
      )
      (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): QuantConv2d(
        16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
        (weight_quant): weight_quantize_fn()
      )
      (conv2

In [71]:
w_bit = 4
weight_q = model.layer1[0].conv1.weight_q # quantized value is stored during the training
w_alpha = model.layer1[0].conv1.weight_quant.wgt_alpha   # alpha is defined in your model already. bring it out here
w_delta = (w_alpha) / (2 ** (w_bit - 1) - 1)    # delta can be calculated by using alpha and w_bit
weight_int = weight_q / w_delta # w_int can be calculated by weight_q and w_delta
print(weight_int) # you should see clean integer numbers

tensor([[[[ 1.0000, -0.0000,  1.0000],
          [ 2.0000, -1.0000,  1.0000],
          [ 3.0000,  5.0000, -1.0000]],

         [[-1.0000, -1.0000, -2.0000],
          [-0.0000, -3.0000, -1.0000],
          [ 1.0000, -1.0000, -2.0000]],

         [[-0.0000,  2.0000,  0.0000],
          [ 1.0000,  1.0000,  3.0000],
          [ 4.0000,  3.0000,  4.0000]],

         ...,

         [[-0.0000,  0.0000,  5.0000],
          [-3.0000,  4.0000,  0.0000],
          [-2.0000, -3.0000,  0.0000]],

         [[-2.0000, -5.0000,  2.0000],
          [-1.0000, -4.0000,  0.0000],
          [-4.0000, -5.0000, -1.0000]],

         [[ 2.0000,  1.0000, -3.0000],
          [ 1.0000, -0.0000,  4.0000],
          [-3.0000,  5.0000, -1.0000]]],


        [[[-1.0000, -0.0000,  0.0000],
          [-0.0000, -2.0000, -2.0000],
          [ 4.0000,  3.0000,  2.0000]],

         [[-2.0000, -1.0000, -1.0000],
          [ 1.0000, -2.0000, -1.0000],
          [ 2.0000, -0.0000,  3.0000]],

         [[-5.0000,  0.0000,  1

In [78]:
x_bit = 4    

x = captured_inputs[0]  # input of the 2nd conv layer

x_alpha = model.layer1[0].conv1.act_alpha
x_delta = (x_alpha) / (2 ** x_bit - 1)

act_quant_fn = act_quantization(x_bit) # define the quantization function
x_q = act_quant_fn(x, x_alpha)         # create the quantized value for x

x_int = x_q / x_delta
print(x_int) # you should see clean integer numbers 

tensor([[[[ 0.0000,  2.0000,  2.0000,  ...,  2.0000,  2.0000,  1.0000],
          [ 0.0000,  3.0000,  3.0000,  ...,  3.0000,  3.0000,  2.0000],
          [ 0.0000,  3.0000,  3.0000,  ...,  2.0000,  3.0000,  2.0000],
          ...,
          [ 2.0000,  0.0000,  0.0000,  ...,  0.0000,  1.0000,  2.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  2.0000,  0.0000],
          [ 1.0000,  2.0000,  2.0000,  ...,  2.0000,  3.0000,  5.0000]],

         [[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          ...,
          [ 5.0000,  7.0000,  8.0000,  ...,  8.0000,  7.0000,  7.0000],
          [ 6.0000,  7.0000,  7.0000,  ...,  8.0000,  7.0000,  7.0000],
          [ 5.0000,  5.0000,  5.0000,  ...,  6.0000,  5.0000,  4.0000]],

         [[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  

In [80]:
conv_int = torch.nn.Conv2d(in_channels = 64, out_channels=64, kernel_size = 3, bias = False)
conv_int.weight = torch.nn.parameter.Parameter(weight_int)

output_int = conv_int(x_int)    # output_int can be calculated with conv_int and x_int
output_recovered = output_int * x_delta * w_delta  # recover with x_delta and w_delta
print(output_recovered)

tensor([[[[ 1.6547e+01,  8.4752e+00,  8.0717e+00,  ...,  1.1300e+01,
            6.6591e+00,  7.0627e+00],
          [ 1.6143e+01, -2.0179e+00,  6.0537e-01,  ...,  8.4752e+00,
            1.0090e+00,  4.8430e+00],
          [ 1.8161e+01,  1.6143e+00,  3.4305e+00,  ...,  1.2915e+01,
            5.2466e+00,  8.4752e+00],
          ...,
          [-9.8878e+00, -1.5395e-06,  1.2511e+01,  ...,  8.2735e+00,
            9.0806e+00, -9.6860e+00],
          [-5.4484e+00,  5.6502e+00,  1.0090e+00,  ...,  4.2376e+00,
            1.3318e+01,  3.2287e+00],
          [-9.0806e+00,  5.6502e+00,  6.0537e-01,  ..., -1.1906e+01,
            6.8609e+00, -8.2735e+00]],

         [[ 3.4305e+01,  3.9148e+01,  4.2578e+01,  ...,  4.3789e+01,
            4.7623e+01,  5.3071e+01],
          [ 3.0874e+01,  3.6726e+01,  4.1569e+01,  ...,  3.3094e+01,
            4.2174e+01,  4.3183e+01],
          [ 3.3497e+01,  3.6322e+01,  4.0157e+01,  ...,  2.7242e+01,
            3.6726e+01,  4.5201e+01],
          ...,
     

In [82]:
conv_ref = torch.nn.Conv2d(in_channels = 64, out_channels=64, kernel_size = 3, bias = False)
conv_ref.weight = model.layer1[0].conv1.weight_q 

output_ref = conv_ref(x)
print(output_ref)    

tensor([[[[ 1.9304e+01,  9.3630e+00,  8.0868e+00,  ...,  1.2013e+01,
            8.2589e+00,  6.1749e+00],
          [ 1.6542e+01,  8.5744e-01,  2.0897e+00,  ...,  9.5757e+00,
            2.5559e+00,  6.0175e+00],
          [ 1.9141e+01,  3.5970e+00,  4.3708e+00,  ...,  1.2302e+01,
            4.4266e+00,  8.2036e+00],
          ...,
          [-1.0406e+01,  4.3200e+00,  1.3251e+01,  ...,  7.8945e+00,
            1.1485e+01, -9.8697e+00],
          [-6.2434e+00,  3.7863e+00,  2.7278e+00,  ...,  2.7459e+00,
            1.0972e+01, -5.8898e-01],
          [-6.7919e+00,  5.2068e+00, -9.9212e-01,  ..., -1.0177e+01,
            7.7885e+00, -1.0156e+01]],

         [[ 3.2563e+01,  4.0081e+01,  4.2609e+01,  ...,  4.3194e+01,
            4.6133e+01,  5.2073e+01],
          [ 3.0161e+01,  3.9276e+01,  4.3962e+01,  ...,  3.2849e+01,
            3.9663e+01,  4.4383e+01],
          [ 3.3443e+01,  3.7279e+01,  4.2964e+01,  ...,  2.5554e+01,
            3.4772e+01,  4.4522e+01],
          ...,
     

In [83]:
difference = abs( output_ref - output_recovered )
print(difference.mean())

tensor(1.2065, device='cuda:0', grad_fn=<MeanBackward0>)


In [84]:
conv_ref = torch.nn.Conv2d(in_channels = 64, out_channels=64, kernel_size = 3, bias = False)
weight = model.layer1[0].conv1.weight
mean = weight.data.mean()
std = weight.data.std()
conv_ref.weight = torch.nn.parameter.Parameter(weight.add(-mean).div(std))

output_ref = conv_ref(x)
print(output_ref)

tensor([[[[ 1.6030e+01,  7.2078e+00,  6.2415e+00,  ...,  1.0016e+01,
            6.1635e+00,  4.6360e+00],
          [ 1.3961e+01, -4.8306e-01,  1.0918e+00,  ...,  8.7287e+00,
            1.6950e+00,  5.4884e+00],
          [ 1.6547e+01,  2.4178e+00,  3.3742e+00,  ...,  1.2561e+01,
            4.1158e+00,  7.9638e+00],
          ...,
          [-1.1194e+01,  2.8705e+00,  1.2131e+01,  ...,  6.4523e+00,
            1.0469e+01, -1.0128e+01],
          [-6.8076e+00,  1.6053e+00,  9.1686e-01,  ...,  1.0486e+00,
            8.9610e+00, -1.7906e+00],
          [-7.1602e+00,  1.9838e+00, -3.9035e+00,  ..., -1.2282e+01,
            6.2060e+00, -1.2674e+01]],

         [[ 3.3144e+01,  4.0250e+01,  4.2870e+01,  ...,  4.4124e+01,
            4.6638e+01,  5.2047e+01],
          [ 3.0041e+01,  3.8930e+01,  4.3417e+01,  ...,  3.3429e+01,
            3.9829e+01,  4.4059e+01],
          [ 3.3874e+01,  3.8407e+01,  4.4053e+01,  ...,  2.6390e+01,
            3.5618e+01,  4.4445e+01],
          ...,
     

In [89]:
difference = abs( output_ref - output_recovered )
print(difference.mean())
print(output_ref.size())

tensor(2.3018, device='cuda:0', grad_fn=<MeanBackward0>)
torch.Size([128, 16, 30, 30])
